# ⚽ Fotbolls-videoanalys med YOLOv8 – Google Colab

Kör cellerna uppifrån och ner.

**Två viktiga saker för att slippa förlora arbete:**
1. Välj GPU: `Körning → Ändra körningstyp → T4 GPU`.
2. Resultat och nedladdad video sparas på **Google Drive** (steg 3), så de överlever även om Colab kopplar ner.

> ⚠️ Colab kopplar ner efter ~90 min inaktivitet. Håll fliken öppen/aktiv under långa körningar (eller använd Colab Pro för längre sessioner).

## 0. Snabbstart (rekommenderas)

Colab tömmer allt vid omstart – både installerade paket, koden i `/content` och Python-minnet. Kör därför **cellen nedan efter varje ny eller omstartad session**, så är allt på plats i ett svep. Sätt sedan `START_SECONDS` i steg 5 och kör den.

Stegen 1–4 nedanför är samma sak uppdelat, om du hellre vill köra dem var för sig.

In [ ]:
# === SNABBSTART: kör denna enda cell efter varje ny/omstartad session ===
# Installerar beroenden, hämtar SENASTE koden, monterar Drive och hittar videon.
!pip install -q ultralytics opencv-python-headless pyyaml tqdm yt-dlp seaborn scipy

import os, sys
REPO = '/content/football-video-analysis'
if not os.path.isdir(REPO):
    !git clone https://github.com/ehoukhe/football-video-analysis.git $REPO
else:
    !cd $REPO && git fetch -q origin && git reset -q --hard origin/main
os.chdir(REPO); sys.path.insert(0, os.getcwd())
# Töm ev. gammal src-kod ur minnet så nyss hämtad kod används.
for _m in [m for m in sys.modules if m == 'src' or m.startswith('src.')]:
    del sys.modules[_m]
!git -C $REPO log --oneline -1

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/fotboll'
DATA_DIR   = os.path.join(DRIVE_ROOT, 'data')
OUTPUT_DIR = os.path.join(DRIVE_ROOT, 'output')
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUTPUT_DIR, exist_ok=True)

from pathlib import Path
_vids = sorted(Path(DATA_DIR).glob('*.mp4'))
video_path = str(_vids[0]) if _vids else None
print('Setup klar. Kod:', os.getcwd())
print('Video:', video_path or f'(ingen än – kör steg 4 eller lägg en .mp4 i {DATA_DIR})')

## 1. Installera beroenden

In [ ]:
!pip install -q ultralytics opencv-python-headless pyyaml tqdm yt-dlp seaborn scipy

## 2. Hämta projektkoden

Klonar det (publika) repot in i Colab-sessionen.

In [ ]:
import os, sys

if not os.path.isdir('/content/football-video-analysis'):
    !git clone https://github.com/ehoukhe/football-video-analysis.git /content/football-video-analysis
else:
    !cd /content/football-video-analysis && git pull

os.chdir('/content/football-video-analysis')
sys.path.insert(0, os.getcwd())
print('Arbetskatalog:', os.getcwd())

## 3. Anslut Google Drive (så resultat & video överlever omstart)

Första gången öppnas ett fönster där du godkänner åtkomst. Video och resultat sparas sedan under `MyDrive/fotboll/`.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')  # idempotent – klagar inte om redan monterad

DRIVE_ROOT = '/content/drive/MyDrive/fotboll'
DATA_DIR = os.path.join(DRIVE_ROOT, 'data')      # nedladdade videor
OUTPUT_DIR = os.path.join(DRIVE_ROOT, 'output')  # heatmaps, statistik, annoterad video
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Video sparas i :', DATA_DIR)
print('Resultat sparas i:', OUTPUT_DIR)

## 4. Välj video

Ange en YouTube-URL eller ladda upp en fil. Finns videon redan på Drive laddas den **inte** ner igen.

In [ ]:
import os, sys
os.chdir('/content/football-video-analysis'); sys.path.insert(0, os.getcwd())  # robust import-väg
from pathlib import Path
from src.video import download_youtube

# Säkerställ Drive-mappar om Snabbstart/steg 3 inte körts i denna session.
if 'DATA_DIR' not in globals():
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/fotboll'
    DATA_DIR   = os.path.join(DRIVE_ROOT, 'data')
    OUTPUT_DIR = os.path.join(DRIVE_ROOT, 'output')
    os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUTPUT_DIR, exist_ok=True)

YOUTUBE_URL = ''  # klistra in en URL här, eller lämna tomt för uppladdning

existing = sorted(Path(DATA_DIR).glob('*.mp4'))
if YOUTUBE_URL:
    if existing:
        video_path = str(existing[0])
        print('Använder redan nedladdad video:', video_path)
    else:
        video_path = download_youtube(YOUTUBE_URL, DATA_DIR)
elif existing:
    video_path = str(existing[0])
    print('Använder befintlig video på Drive:', video_path)
else:
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    video_path = os.path.join(DATA_DIR, name)
    os.replace(name, video_path)

print('Video:', video_path)

## 5. Kör analysen

Använder presetten **`config/accuracy.yaml`**: stor modell (`yolov8l`), hög upplösning så bortre spelare syns, BoT-SORT, separata boll/spelar-trösklar och **automatisk lag-detektering**.

> ⚠️ **Hoppa över introt!** YouTube-matcher börjar ofta med uppställning/ceremoni. Sätt `START_SECONDS` till ungefär när avsparken sker (t.ex. `180` = 3 min in), annars analyserar snabbtestet bara förspelet — då blir possession meningslös (domaren håller bollen).

Börja med ett **snabbtest** (`max_frames`); sätt `0` för hela matchen när du är nöjd.

In [ ]:
import os, sys
os.chdir('/content/football-video-analysis'); sys.path.insert(0, os.getcwd())  # robust import-väg
from src.pipeline import analyze_video, load_config

START_SECONDS = 180   # ~när avsparken sker – hoppar över uppställning/ceremoni. Justera per match!

cfg = load_config('config/accuracy.yaml')  # de framtestade noggrannhets-inställningarna
cfg['model']['device'] = '0'                # GPU
cfg['output']['dir'] = OUTPUT_DIR           # spara resultat på Drive
cfg['video']['start_seconds'] = START_SECONDS
cfg['video']['max_frames'] = 1500           # snabbtest; sätt 0 för hela matchen
# cfg['video']['frame_stride'] = 4          # ~4x snabbare för långa körningar

result = analyze_video(video_path, cfg)
print('Klar! Resultat sparat i', OUTPUT_DIR)

## 6. Resultat: statistik och insikter

In [ ]:
import json
print(json.dumps(result.stats.summary(), ensure_ascii=False, indent=2))
print('\n=== Insikter till tränaren ===')
for i, ins in enumerate(result.insights, 1):
    print(f'{i}. {ins}')

## 7. Visa heatmaps

In [ ]:
from IPython.display import Image, display
for hm in result.heatmaps:
    display(Image(filename=hm))

## 8. Resultatfilerna

Allt ligger redan kvar på din Google Drive under `MyDrive/fotboll/output/` (annoterad video, heatmaps, `*_stats.json`) – även om Colab kopplar ner. Kör cellen nedan för att lista dem.

In [ ]:
for f in sorted(Path(OUTPUT_DIR).iterdir()):
    print(f.name, f'({f.stat().st_size/1e6:.1f} MB)')

## 9. Fas 2 – Automatisk plankalibrering (top-down & meter)

Räknar om homografin **per bildruta** via en tränad plan-nyckelpunktsmodell (YOLOv8-pose, 32 punkter), så att en rörlig kamera (XbotGo/sändning) kan mappas till planen. Ger:
- **Top-down-heatmaps** på en planritning
- **Löpsträckor i meter** (inte pixlar)

Modellen körs i **ultralytics på GPU** (samma ramverk som resten) och laddas ner automatiskt från HuggingFace — **ingen API-nyckel behövs**. Kräver bara att du kört Snabbstart + steg 4 först.

> Detta är nytt och behöver valideras på din film. Kolla `*_topdown_players.png`: ligger spelarna på rimliga platser på planen? Om punkterna ser **hopblandade/förvridna** ut kan modellens nyckelpunkts-ordning skilja sig från vår (då remappar vi). Ser den bara **gles/tom** ut (`calibrated: False`) hittar modellen för få punkter i din kameravinkel – sänk `conf` (t.ex. 0.3). Berätta vad du ser.

In [ ]:
# === FAS 2: automatisk plankalibrering (per-ruta homografi) ===
# Kör fält-modellen via ultralytics (.pt) på GPU – inga tunga externa beroenden,
# ingen Roboflow-nyckel, ingen torch/onnx-krock.
import os, sys
REPO = '/content/football-video-analysis'
!cd $REPO && git fetch -q origin && git reset -q --hard origin/main && git log --oneline -1
os.chdir(REPO); sys.path.insert(0, os.getcwd())
for _m in [m for m in sys.modules if m == 'src' or m.startswith('src.')]:
    del sys.modules[_m]

from pathlib import Path
if 'OUTPUT_DIR' not in globals():
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/fotboll'
    DATA_DIR   = os.path.join(DRIVE_ROOT, 'data')
    OUTPUT_DIR = os.path.join(DRIVE_ROOT, 'output')
    os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUTPUT_DIR, exist_ok=True)
if 'video_path' not in globals() or not video_path:
    _v = sorted(Path(DATA_DIR).glob('*.mp4'))
    video_path = str(_v[0]) if _v else None
assert video_path, 'Ingen video – kör Snabbstart/steg 4 först.'

# Ladda ner plan-keypoint-modellen (YOLOv8-pose, 32 punkter) en gång.
KP_WEIGHTS = '/content/pitch_keypoints.pt'
if not os.path.exists(KP_WEIGHTS):
    !wget -q -O $KP_WEIGHTS https://huggingface.co/martinjolif/yolo-football-pitch-detection/resolve/main/yolo-football-pitch-detection.pt
assert os.path.getsize(KP_WEIGHTS) > 1_000_000, 'Nedladdning av keypoint-modellen misslyckades.'

from ultralytics import YOLO
from src.pitch import YoloPitchKeypointDetector, soccer_vertices_m, AutoKeypointCalibrator
from src.pipeline import analyze_video, load_config

START_SECONDS = 180            # avspark – justera per match

kp_model = YOLO(KP_WEIGHTS)
# Detektor: plan-nyckelpunkter -> plankoordinater i meter (105 x 68) på GPU.
field_detect = YoloPitchKeypointDetector(
    kp_model, soccer_vertices_m(105, 68), conf=0.5, imgsz=1280, device='0'
)
# recompute_every=15: kör fält-modellen var 15:e ruta, återanvänd homografin emellan.
calibrator = AutoKeypointCalibrator(field_detect, min_points=4, recompute_every=15)

cfg = load_config('config/accuracy.yaml')
cfg['model']['device'] = '0'
cfg['output']['dir'] = OUTPUT_DIR
cfg['video']['start_seconds'] = START_SECONDS
cfg['video']['max_frames'] = 800

result = analyze_video(video_path, cfg, calibrator=calibrator)
print('calibrated:', result.stats.summary().get('calibrated'))
for hm in result.heatmaps:
    print('heatmap:', hm)